# Trabajo Práctico 2 (TP2) - Parte 1: Preprocesamiento de Datos y Análisis Estadístico Exploratorio

### Machine Learning 1 (23433)
#### Facultad de Ingeniería - Universidad Nacional de Asunción (FIUNA)

---

## Objetivos de la Parte 1
1. Cargar e inspeccionar la estructura del conjunto de datos unificado de rendimiento académico de FIUNA (`reglamento_nuevo_unificado.csv`).
2. Mapear las 27 siglas e intensificaciones curriculares a las 7 carreras principales de la facultad.
3. Aplicar un preprocesamiento de integridad de datos **no destructivo** que preserve el 100% de los 64,295 registros sin eliminar filas.
4. Responder a 6 preguntas estadísticas exploratorias clave para auditar la masa estudiantil, la retención académica y las tasas de aprobación.


In [1]:
# 1. Carga de Librerías Fundamentales
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)

# Carga del Dataset
csv_path = 'datos_concatenados.csv'
if not os.path.exists(csv_path):
    csv_path = os.path.join('..', 'Clase_5', 'reglamento_nuevo_unificado.csv')

df_raw = pd.read_csv(csv_path)
print(f"Dataset cargado exitosamente: {df_raw.shape[0]:,} filas y {df_raw.shape[1]} columnas.")


Dataset cargado exitosamente: 64,295 filas y 31 columnas.


## Ejercicio 1: Mapeo de Intensificaciones Curriculares a Carreras Principales

Completa el diccionario `career_code_mapping` para mapear las 27 siglas del sistema (`CIV-PLS13`, `INT9CONSTR`, `ELE-PLS23`, `INT9SDIGYT`, `MCT-PLS13`, `IND-PLS13`, `CGF-PLS13`, `MEC-PLS13`, `ECA-PLS13`, etc.) a sus 7 carreras principales:
- `Ing. Civil`
- `Ing. Electrónica`
- `Ing. Mecatrónica`
- `Ing. Industrial`
- `Ing. Geográfica`
- `Ing. Mecánica`
- `Ing. Electromecánica`


In [2]:
# TODO: Completar la estructura del diccionario career_code_mapping
career_code_mapping = {
    # === ESCRIBE TU CÓDIGO AQUÍ ===
    'CIV-PLS13': 'Ing. Civil', 'CIV-PLS23': 'Ing. Civil', 'INT9CONSTR': 'Ing. Civil',
    'INT9TRANSP': 'Ing. Civil', 'INT9ORTERR': 'Ing. Civil', 'INT9SANEHI': 'Ing. Civil',
    
    'ELE-PLS13': 'Ing. Electrónica', 'ELE-PLS23': 'Ing. Electrónica',
    'INT9ELECTR': 'Ing. Electrónica', 'INT9SDIGYT': 'Ing. Electrónica',
    
    'MCT-PLS13': 'Ing. Mecatrónica', 'MCT-PLS23': 'Ing. Mecatrónica', 'MCT9-OPT': 'Ing. Mecatrónica',
    
    'IND-PLS13': 'Ing. Industrial', 'IND-PLS23': 'Ing. Industrial',
    'INT9G-ECO': 'Ing. Industrial', 'INT9-PROYT': 'Ing. Industrial',
    
    'CGF-PLS13': 'Ing. Geográfica', 'CGF-PLS23': 'Ing. Geográfica', 'INT9RNYMA': 'Ing. Geográfica',
    
    'MEC-PLS13': 'Ing. Mecánica', 'MEC-PLS23': 'Ing. Mecánica',
    'INT9MECANI': 'Ing. Mecánica', 'MEC9-OPT': 'Ing. Mecánica',
    
    'ECA-PLS13': 'Ing. Electromecánica', 'ECA-PLS23': 'Ing. Electromecánica', 'ECA9-OPT': 'Ing. Electromecánica'
}

df_clean = df_raw.copy()
# TODO: Asignar la nueva columna Carrera_Nombre usando map()
df_clean['Carrera_Nombre'] = df_clean['Cod.Car.Sec'].astype(str).str.strip().map(career_code_mapping)

# Definición del Target Binario (1 para 'S', 0 para 'N')
df_clean = df_clean.dropna(subset=['Aprobado', 'Carrera_Nombre']).copy()
df_clean['Target'] = (df_clean['Aprobado'] == 'S').astype(int)

print(f"Filas tras mapeo de carreras: {len(df_clean):,}")


Filas tras mapeo de carreras: 64,295


## Ejercicio 2: Limpieza Numérica y Generación de Atributos Derivados (Sin Eliminación de Filas)

Convierte los campos numéricos de parciales y evaluaciones a formato float utilizando `pd.to_numeric(..., errors='coerce')` para mantener el 100% de las filas.
Crea las siguientes variables derivadas:
- `Score_Parciales`: Promedio entre el 1er y 2do parcial.
- `Diff_Parciales`: Diferencia ($2^\circ\text{Par} - 1^\circ\text{Par}$).


In [ ]:
# TODO: Limpiar y convertir atributos numéricos sin eliminar filas
# === ESCRIBE TU CÓDIGO AQUÍ ===
df_clean['Primer_Par_Clean'] = pd.to_numeric(df_clean['Primer.Par'], errors='coerce')
df_clean['Segundo_Par_Clean'] = pd.to_numeric(df_clean['Segundo.Par'], errors='coerce')
df_clean['Score_Parciales'] = (df_clean['Primer_Par_Clean'].fillna(0) + df_clean['Segundo_Par_Clean'].fillna(0)) / 2.0
df_clean['Diff_Parciales'] = df_clean['Segundo_Par_Clean'].fillna(0) - df_clean['Primer_Par_Clean'].fillna(0)
df_clean['TPLab_Clean'] = pd.to_numeric(df_clean['TPLab.'], errors='coerce')
df_clean['Asis_Clean'] = pd.to_numeric(df_clean['Asis'], errors='coerce')
df_clean['Firma_Clean'] = pd.to_numeric(df_clean['Firma'], errors='coerce')
df_clean['FirmaCalc_Clean'] = pd.to_numeric(df_clean['FirmaCalculada'], errors='coerce')

print(f"Filas preservadas en df_clean: {len(df_clean):,} (100% retención)")


Filas preservadas en df_clean: 64,295 (100% retención)


## Ejercicio 3: Auditoría Estadísticas Exploratoria (6 Preguntas Clave)

Responde a las siguientes 6 preguntas estadísticas utilizando código en Python sobre `df_raw` y `df_clean`:

1. **¿Cuántos estudiantes presentan registros en los tres ciclos del CSV?**
2. **¿Cuántas personas/registros tienen calificación final (`Nota.Final`)?**
3. **¿Cuántos tienen proceso (`FirmaCalculada` / `Firma`)?**
4. **¿Cuál es la tasa de aprobación global y por Carrera?**
5. **¿Cómo se comparan las notas medias del 1er y 2do Parcial según condición final (Aprobado vs No Aprobado)?**
6. **¿Cuál es la tasa de abandono / inasistencia total a parciales?**


In [16]:
df_clean['Ciclo'] = df_clean['Anho'].astype(str).str.strip() + '-' + df_clean['Semestre'].astype(str).str.strip()
ciclos_por_estudiante = df_clean.groupby('ALUMNO_ID')['Ciclo'].nunique()
estudiantes_con_3_ciclos = (ciclos_por_estudiante == 3).sum()
print(f"1. Estudiantes con exactamente 3 ciclos: {estudiantes_con_3_ciclos:,} de {len(ciclos_por_estudiante):,} ({estudiantes_con_3_ciclos / len(ciclos_por_estudiante) * 100:.2f}%)")



1. Estudiantes con exactamente 3 ciclos: 3,532 de 4,759 (74.22%)


In [15]:
estudiantes_con_notaFinal = df_clean[df_clean['Nota.Final'].notna()]['ALUMNO_ID'].nunique()
registros_con_notaFinal = df_clean[df_clean['Nota.Final'].notna()].shape[0]
print(f"2. Estudiantes con Nota Final: {estudiantes_con_notaFinal:,} de {len(ciclos_por_estudiante):,} ({estudiantes_con_notaFinal / len(ciclos_por_estudiante) * 100:.2f}%)")
print(f"Registros con Nota Final: {registros_con_notaFinal:,} de {df_clean.shape[0]:,} ({registros_con_notaFinal / df_clean.shape[0] * 100:.2f}%)")

2. Estudiantes con Nota Final: 4,344 de 4,759 (91.28%)
Registros con Nota Final: 42,720 de 64,295 (66.44%)


In [14]:
registros_con_firmacalculada = df_clean['FirmaCalc_Clean'].notna().sum()
registros_con_firma_50 = (df_clean['Firma_Clean']>=50).sum()
porcentaje_firma_50 = (registros_con_firma_50 / df_clean.shape[0]) * 100
porcentaje_firma_calculada = (registros_con_firmacalculada / df_clean.shape[0]) * 100
print(f"3.Registros con Firma Calculada: {registros_con_firmacalculada:,} de {df_clean.shape[0]:,} ({porcentaje_firma_calculada:.2f}%)")
print(f"Registros con Firma >= 50: {registros_con_firma_50:,} de {df_clean.shape[0]:,} ({porcentaje_firma_50:.2f}%)")

3.Registros con Firma Calculada: 19,407 de 64,295 (30.18%)
Registros con Firma >= 50: 41,715 de 64,295 (64.88%)


In [12]:
tasa_aprobacion_global = df_clean['Target'].mean() * 100
tasa_por_carrera = df_clean.groupby('Carrera_Nombre').agg({
    'Target': ['count', 'sum', 'mean']
}).round(4)
tasa_por_carrera.columns = ['Inscritos', 'Aprobados', 'Tasa_Aprob']
tasa_por_carrera['Tasa_Aprob_Pct'] = tasa_por_carrera['Tasa_Aprob'] * 100
print(f"Tasa de Aprobación Global: {tasa_aprobacion_global:.2f}%\n")
print("4. Tasa de Aprobación por Carrera:")
print(tasa_por_carrera[['Inscritos', 'Aprobados', 'Tasa_Aprob_Pct']])

Tasa de Aprobación Global: 60.67%

4. Tasa de Aprobación por Carrera:
                      Inscritos  Aprobados  Tasa_Aprob_Pct
Carrera_Nombre                                            
Ing. Civil                29392      17202           58.53
Ing. Electromecánica       1693       1150           67.93
Ing. Electrónica          11928       6638           55.65
Ing. Geográfica            4626       3275           70.80
Ing. Industrial            6416       4050           63.12
Ing. Mecatrónica           7147       4691           65.64
Ing. Mecánica              3093       1999           64.63


In [13]:
promedios_parciales = df_clean.groupby('Target')[['Primer_Par_Clean', 'Segundo_Par_Clean']].mean()
promedios_parciales.index = ['No Aprobado', 'Aprobado']
print("5. Promedios de Parciales por Condición Final:")
print(promedios_parciales.round(2))
diferencia = promedios_parciales.loc['Aprobado'] - promedios_parciales.loc['No Aprobado']
print("\nDiferencia (Aprobado - No Aprobado):")
print(diferencia.round(2))

5. Promedios de Parciales por Condición Final:
             Primer_Par_Clean  Segundo_Par_Clean
No Aprobado             26.16              19.56
Aprobado                63.63              65.53

Diferencia (Aprobado - No Aprobado):
Primer_Par_Clean     37.47
Segundo_Par_Clean    45.97
dtype: float64


In [17]:
abandono = (df_clean['Primer_Par_Clean'].fillna(0) == 0) & (df_clean['Segundo_Par_Clean'].fillna(0) == 0)
registros_abandono = abandono.sum()
total_registros = len(df_clean)
porcentaje_abandono = (registros_abandono / total_registros) * 100
print(f"6. Registros con 0 en ambos parciales (abandono): {registros_abandono:,} ({porcentaje_abandono:.2f}%)")

6. Registros con 0 en ambos parciales (abandono): 9,340 (14.53%)
